# Tutorial: CNN 01 Convolution From Scratch

Audience:
- Learners moving from dense neural networks to convolutional structure.

Prerequisites:
- Linear algebra, basic Python, and the Module 06 discussion of affine layers.

Learning goals:
- Implement 1D and 2D cross-correlation directly.
- Verify the matrix-multiplication view of convolution.
- Visualize how hand-designed filters produce feature maps.


## Outline

1. Build 1D convolution by explicit loops.
2. Re-express the same operation as matrix multiplication.
3. Extend the idea to 2D image patches.
4. Visualize edge and blur filters on a real digit image.


In [ ]:
from __future__ import annotations

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

SEED = 21
random.seed(SEED)
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (6, 4)
SEED


## Step 1 - Implement 1D cross-correlation

Deep-learning libraries usually implement cross-correlation rather than the filter-reversed mathematical convolution. That distinction is harmless once the weights are learned.


In [ ]:
def conv1d_valid(x: np.ndarray, w: np.ndarray, stride: int = 1) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    outputs = []
    for start in range(0, len(x) - len(w) + 1, stride):
        patch = x[start : start + len(w)]
        outputs.append(np.dot(patch, w))
    return np.array(outputs)

x = np.array([1, 2, 0, -1, 3, 1], dtype=float)
w = np.array([2, -1, 1], dtype=float)
conv1d_valid(x, w)


## Step 2 - Matrix view

A convolution layer is still a linear map. The difference from a dense layer is that the corresponding matrix has repeated diagonals and many zeros.


In [ ]:
def conv1d_matrix(kernel: np.ndarray, input_length: int, stride: int = 1) -> np.ndarray:
    kernel = np.asarray(kernel, dtype=float)
    rows = []
    for start in range(0, input_length - len(kernel) + 1, stride):
        row = np.zeros(input_length)
        row[start : start + len(kernel)] = kernel
        rows.append(row)
    return np.vstack(rows)

T = conv1d_matrix(w, len(x))
loop_output = conv1d_valid(x, w)
matrix_output = T @ x
T, loop_output, matrix_output, np.allclose(loop_output, matrix_output)


## Step 3 - Implement 2D cross-correlation

Now we move from signals to images. Each output entry is a dot product between a local patch and a filter.


In [ ]:
def conv2d_valid(image: np.ndarray, kernel: np.ndarray, stride: int = 1) -> np.ndarray:
    image = np.asarray(image, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    kh, kw = kernel.shape
    rows = []
    for i in range(0, image.shape[0] - kh + 1, stride):
        row = []
        for j in range(0, image.shape[1] - kw + 1, stride):
            patch = image[i : i + kh, j : j + kw]
            row.append(np.sum(patch * kernel))
        rows.append(row)
    return np.array(rows)

sample = np.arange(1, 17, dtype=float).reshape(4, 4)
edge_kernel = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=float)
conv2d_valid(sample, edge_kernel)


## Step 4 - Patch matrix view in 2D

The `im2col` idea flattens each local patch into one row. Multiplying that patch matrix by a flattened kernel gives the same answer as the explicit nested loops.


In [ ]:
def image_to_patches(image: np.ndarray, kernel_shape: tuple[int, int], stride: int = 1) -> np.ndarray:
    kh, kw = kernel_shape
    patches = []
    for i in range(0, image.shape[0] - kh + 1, stride):
        for j in range(0, image.shape[1] - kw + 1, stride):
            patches.append(image[i : i + kh, j : j + kw].reshape(-1))
    return np.vstack(patches)

patch_matrix = image_to_patches(sample, edge_kernel.shape)
flat_kernel = edge_kernel.reshape(-1)
patch_output = (patch_matrix @ flat_kernel).reshape(conv2d_valid(sample, edge_kernel).shape)
patch_output, np.allclose(patch_output, conv2d_valid(sample, edge_kernel))


## Step 5 - Visualize feature maps on a real image

We use the `sklearn` digits dataset so the notebook runs without downloads. The same ideas scale to larger image collections.


In [ ]:
digits = load_digits()
image = digits.images[0]
label = digits.target[0]

kernels = {
    "vertical edge": np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=float),
    "horizontal edge": np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=float),
    "blur": np.ones((3, 3), dtype=float) / 9.0,
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float),
}

feature_maps = {name: conv2d_valid(image, kernel) for name, kernel in kernels.items()}

fig, axes = plt.subplots(1, len(feature_maps) + 1, figsize=(14, 3))
axes[0].imshow(image, cmap="gray")
axes[0].set_title(f"digit={label}")
axes[0].axis("off")

for ax, (name, fmap) in zip(axes[1:], feature_maps.items()):
    ax.imshow(fmap, cmap="coolwarm")
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()


## Step 6 - What the visualizations mean

The edge filters respond strongly where pixel intensity changes in their preferred direction. The blur filter smooths local noise, while the sharpen filter emphasizes contrast. Learned CNN filters start from random weights, but they often converge toward similarly interpretable local detectors in their first layers.


In [ ]:
parameter_example = {
    "dense_32x32x3_to_64": 32 * 32 * 3 * 64,
    "conv_64_filters_3x3_rgb": 64 * 3 * 3 * 3,
}
parameter_example


## Exercises

- Change the stride in `conv1d_valid` and `conv2d_valid` and verify the output shapes.
- Replace the hand-designed kernels with random `3x3` filters and inspect the resulting feature maps.
- Add zero padding and verify how the matrix view changes.
